In [ ]:
import os, re, subprocess, glob, time, sys
t0 = time.time()
OUT = '/kaggle/working/repo'
os.makedirs(OUT, exist_ok=True)
os.chdir(OUT)

# ---------------------------------------------------------------- данные
# Предобработанные окна лежат в выводе ядра lead3. Путь к примонтированному
# выводу зависит от слага, поэтому ищем по X.dat, а не хардкодим.
print('в /kaggle/input:', os.listdir('/kaggle/input'), flush=True)
cands = sorted(glob.glob('/kaggle/input/**/ADFTD/X.dat', recursive=True))
print('найдено X.dat:', cands, flush=True)
if not cands:
    raise SystemExit('не найден ADFTD/X.dat во входных данных')
SRC = os.path.dirname(cands[0])

# Симлинк вместо копирования: X.dat ~5 ГБ, memmap читается только на чтение.
# MultiDataset склеивает путь как root_path + имя набора, то есть ждёт
# ./Processed/ADFTD (каталог L400 в пути не участвует).
os.makedirs('Processed/L400', exist_ok=True)
for link in ('Processed/ADFTD', 'Processed/L400/ADFTD'):
    if not os.path.exists(link):
        os.symlink(SRC, link)
print(open('Processed/ADFTD/meta.json').read()[:220], flush=True)

# ---------------------------------------------------------------- код LEAD
subprocess.run('pip install -q huggingface_hub einops', shell=True)
subprocess.run('git clone -q https://github.com/DL4mHealth/LEAD.git', shell=True)

# В adftd_loader.py две функции с именем get_id_list_adftd. Побеждает вторая,
# она работает с memmap-данными и умеет только fixed/mccv/loso. Ветки 5-fold
# в ней нет, хотя в первой функции того же файла она есть. Переносим её один
# в один, чтобы каждый из 88 человек ровно один раз попал в тест.
P = 'LEAD/data_provider/dataset_loader/adftd_loader.py'
s = open(P).read()
OLD = """    else:
        raise ValueError('Invalid cross_val. Please use fixed, mccv, or loso.')


class ADFTDLoader(BaseLoader):"""
NEW = """    elif args.cross_val == '5-fold':
        random.seed(42)
        random.shuffle(hc_list)
        random.shuffle(ad_list)
        random.shuffle(ftd_list)
        fh, fa, ff = len(hc_list) / 5, len(ad_list) / 5, len(ftd_list) / 5
        k = (args.seed - 41) % 5
        test_ids = (hc_list[int(k * fh):int((k + 1) * fh)] +
                    ad_list[int(k * fa):int((k + 1) * fa)] +
                    ftd_list[int(k * ff):int((k + 1) * ff)])
        if k == 4:
            val_ids = hc_list[:int(fh)] + ad_list[:int(fa)] + ftd_list[:int(ff)]
        else:
            val_ids = (hc_list[int((k + 1) * fh):int((k + 2) * fh)] +
                       ad_list[int((k + 1) * fa):int((k + 2) * fa)] +
                       ftd_list[int((k + 1) * ff):int((k + 2) * ff)])
        train_ids = [sid for sid in all_ids
                     if sid not in test_ids and sid not in val_ids]
        return sorted(all_ids), sorted(train_ids), sorted(val_ids), sorted(test_ids)

    else:
        raise ValueError('Invalid cross_val. Please use fixed, mccv, 5-fold, or loso.')


class ADFTDLoader(BaseLoader):"""
assert s.count(OLD) == 1, 'не найден ожидаемый блок в adftd_loader.py'
open(P, 'w').write(s.replace(OLD, NEW))
print('5-fold добавлен в memmap-загрузчик', flush=True)

# ---------------------------------------------------------- чекпойнт
# P-11: ADFTD входит в 5 downstream-наборов и НЕ входит в 11 предобучающих,
# поэтому утечки нет. Вариант LEAD-All брать нельзя - он видел FTD-субъектов.
CK = 'checkpoints/LEAD/pretrain/LEAD/P-11-b1024-p50-g16-learnable-3D/nh8_el12_dm128_df256_seed41/checkpoint.pth'
os.makedirs(os.path.dirname(CK), exist_ok=True)
from huggingface_hub import hf_hub_download
hf_hub_download(repo_id='YiheWang/LEAD', filename=CK, local_dir='.')
print('checkpoint', round(os.path.getsize(CK) / 1e6, 1), 'MB', flush=True)

# ---------------------------------------------------------- дообучение
# Было: mccv, itr=1, 2 эпохи, тест 10 человек -> разброс не оценивался вообще.
# Стало: 5-fold, itr=5 (seed 41..45 -> фолды 0..4), 8 эпох, patience=3.
LOG = '/kaggle/working/lead_multi.log'
CMD = (
    'python LEAD/run.py --method LEAD --task_name finetune --is_training 1 '
    '--root_path ./Processed --data_path L400 --data MultiDatasets '
    '--training_dataset ADFTD --model LEAD '
    '--model_id P-11-b1024-p50-g16-learnable-3D-F-ADFTD-multi '
    '--checkpoints_path ./' + CK + ' '
    '--e_layers 12 --batch_size 64 --n_heads 8 --d_model 128 --d_ff 256 '
    '--patch_len 50 --stride 50 --temporal_pos_type learnable --channel_pos_type 3D '
    '--use_sampling_embedding --sampling_rate_list 200,100,50 '
    '--group_shuffle --group_size 4 --use_subject_loss --use_subject_vote '
    '--classify_choice multi_class --cross_val 5-fold --itr 5 '
    '--train_epochs 8 --patience 3 --learning_rate 0.0001 '
    '--seq_len 400 --enc_in 19 --des 5fold_multi '
    '; exit ${PIPESTATUS[0]}'
)
print(CMD, flush=True)
with open(LOG, 'w') as fh:
    p = subprocess.Popen(CMD, shell=True, executable='/bin/bash',
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        fh.write(line)
        fh.flush()
        if re.search(r'Epoch|results|ids:|subjects|Error|Traceback|Confusion|\|', line):
            print(line.rstrip()[:300], flush=True)
rc = p.wait()
print('return code', rc, '| elapsed', round((time.time() - t0) / 60, 1), 'min', flush=True)

# сводка в отдельный файл, чтобы не качать весь вывод ядра
tail = open(LOG).read()
with open('/kaggle/working/summary_multi.txt', 'w') as f:
    for line in tail.splitlines():
        if re.search(r'results ---|Confusion|test ids:|Number of (test|training|validation) subjects', line):
            f.write(line + chr(10))
print(open('/kaggle/working/summary_multi.txt').read()[-3000:])
if rc != 0:
    raise RuntimeError('LEAD run.py завершился с кодом %d' % rc)
